# Predict the estimated days a product will take to be actually delivered 

In [1]:
import pandas as pd, numpy as np
from glob import glob
import time
import os



from pandas.api.types import is_string_dtype, is_numeric_dtype, is_categorical_dtype
from fastai.tabular.all import *
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from IPython.display import Image, display_svg, SVG
from fastai.imports import *

pd.options.display.max_rows = 20
pd.options.display.max_columns = 8
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 150)
matplotlib.rcParams['font.size'] = 14
matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['figure.facecolor'] = '#00000000'

# GUI

# All Functions

In [2]:
def make_new_columns(df):
    df['shipping_duration_days'] =  (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
    df['product_volume_cm3'] =  df['product_length_cm'] * df['product_weight_g'] *  df['product_height_cm']
    
    return df

def load_df_and_clean():
    df = pd.read_csv('items.csv')
    df = df.drop(columns=['Unnamed: 0'])
    to_keep = [
        "price",
        "product_weight_g",

        'total_order_price_value', 
        'total_order_freight_value',
        'total_order_value',

        "product_length_cm",
        "product_height_cm",
        "product_width_cm",

        'order_item_id', # new
        "product_category_name",

        "seller_state",
        "customer_state",
        "seller_city",
        "customer_city",

        "order_purchase_timestamp",
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'freight_value'
    ]

    df = df[to_keep]
    
    # remove the rows that contain null values in any of the following columns
   
    df = df.dropna(subset=['product_category_name','product_weight_g','product_length_cm'
                       ,'product_height_cm' ,'product_width_cm','order_approved_at','order_delivered_carrier_date',
                       'order_delivered_customer_date'
                      ])
    
    
    df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
    df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
    df['order_approved_at'] = pd.to_datetime(df['order_approved_at'])
    df['order_delivered_carrier_date'] = pd.to_datetime(df['order_delivered_carrier_date'])
    
    df['purchase_date'] = df['order_purchase_timestamp'].dt.date
    df['purchase_time'] = df['order_purchase_timestamp'].dt.hour
    
    
    df = make_new_columns(df)
    
    # i will drop the following: 
        # - order_delivered_customer_date
        # - order_approved_at
        # - order_delivered_carrier_date
    # why ? because i the user, mainly the customer won't know the estimated time 
    
    df = df.drop(columns=['order_purchase_timestamp','order_delivered_customer_date', 'total_order_price_value', 
        'total_order_freight_value','order_approved_at',
        'order_delivered_carrier_date'])
    return df
    
    

# make a log transformation quickly and efficiently
def log_transform(df,col):
    df[col]= np.log1p(df[col])
    print('Log Transform successful ✅')    
    return df
    
def set_range_col(df,col,l,r):
    # very cool state of the pledged money raised
    df =  df[(df[col] >= l) & (df[col] <= r)]
    print( df[(df[col] >= l) & (df[col] <= r)][col].skew())
    print('Range successful ✅')    
    return df

def deal_with_outliers(df,col):
    Q1 = df[col].quantile(0.25)  # 25th percentile
    Q3 = df[col].quantile(0.75)  # 75th percentile
    IQR = Q3 - Q1                      # Interquartile range

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    upper_bound = np.ceil(upper_bound)
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    df.loc[df[col] > upper_bound, col] = upper_bound
    print('Outliers successful ✅')
    return df

def train_test_split(df):
    total = df.shape[0]
    testing_portion = int(total * 0.10) # 10% test
    
    # the splits
    training_data = total - testing_portion
    test_data = training_data + testing_portion


    df1 = df[:training_data]
    df_test = df[training_data:test_data]
    return df1, df_test
    
    
def all_data_preprocessing(df):
    
   
    df = deal_with_outliers(df,'total_order_value')
    df = deal_with_outliers(df,'price')
    df = deal_with_outliers(df,'freight_value')
    df = deal_with_outliers(df,'order_item_id')





    df = deal_with_outliers(df,'product_weight_g')
    df = deal_with_outliers(df,'product_length_cm')
    df = deal_with_outliers(df,'product_height_cm')
    df = deal_with_outliers(df,'product_width_cm')

    df = deal_with_outliers(df,'shipping_duration_days')
    df = deal_with_outliers(df,'product_volume_cm3')





    df = log_transform(df,'price')
    df = log_transform(df,'freight_value')
    df = log_transform(df,'order_item_id')

    df = log_transform(df,'total_order_value')

    df = log_transform(df,'product_weight_g')
    df = log_transform(df,'product_length_cm')
    df = log_transform(df,'product_height_cm')
    df = log_transform(df,'product_width_cm')

    df = log_transform(df,'shipping_duration_days')
    df = log_transform(df,'product_volume_cm3')



    
    
    return df


import math
def r_mse(pred, y): return round(math.sqrt(((pred-y)**2).mean()), 6)
def m_rmse(m,xs,y): return r_mse(m.predict(xs),y)



def one_prediction(df):
    dl_test = to.dataloaders().test_dl(df)
    xs_test, y_test = dl_test.train.xs, dl_test.train.y
    xs_test = xs_test[xs.columns.to_list()]
    dl_test_nn = learn.dls.test_dl(df)
    return xs_test,dl_test_nn,y_test

def final_one_prediction(df,rf,xgbR, nn):
    xs_test,nn_xs_y, y_test= one_prediction(df)
    
    pred_rf = rf.predict(xs_test)
    pred_xgb = xgbR.predict(xs_test)
    
    preds_nn , _ = learn.get_preds(dl=nn_xs_y)
    nn_general_prediction = to_np(preds_nn.squeeze())
    
    
    combined_predictions = [
    pred_rf * 0.3,
    pred_xgb * 0.5,
    nn_general_prediction * 0.2
        ]


    # Compute the weighted average
    preds_weighted = np.sum(combined_predictions, axis=0)

    # print(f'Predicted Pledged: $ {np.exp(preds_weighted[0]):.2f}')
    return np.int64(np.round(np.exp(preds_weighted)))


# Real Talk

In [3]:
df = pd.read_csv('items.csv')

In [4]:
main_df = df[[ 'price', 'freight_value',
       'total_order_price_value', 'total_order_freight_value',
       'total_order_value', 'seller_city', 'seller_state',
       'product_category_name', 'product_name_lenght','product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'order_purchase_timestamp',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_city', 'customer_state']]

### load and clean 

In [5]:
df = load_df_and_clean()

### Train / Test Split

In [6]:
def reload():
    global df, df1, df_test, to, y, valid_y, xs, valid_xs, learn, test_df

    df = load_df_and_clean()

    df1, df_test = train_test_split(df)

    to = load_pickle('./to_olist.pkl')
    y = to.train.y
    valid_y = to.valid.y
    xs = load_pickle('./xs_final.pkl')
    valid_xs = load_pickle('./valid_xs_final.pkl')

    learn = load_pickle('./learn.pkl')
    test_df = load_pickle('./test_df.pkl')

# LOAD PICKLES 🥒

In [7]:
learn = load_pickle('./learn.pkl')
test_df = load_pickle('./test_df.pkl')
to = load_pickle('./to_olist.pkl')


### GUI

# Predicting the Number of shipping days that will be taken ⛴️📦

In [8]:
import joblib
learners = joblib.load("olist_learners.joblib")
rf = learners['rf']
xgbR = learners['xgb']
nn = learners['nn']

In [9]:
df.columns.tolist()

['price',
 'product_weight_g',
 'total_order_value',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'order_item_id',
 'product_category_name',
 'seller_state',
 'customer_state',
 'seller_city',
 'customer_city',
 'freight_value',
 'purchase_date',
 'purchase_time',
 'shipping_duration_days',
 'product_volume_cm3']

In [10]:
reload() # reload
# product_length_cm = sorted(set(round(val, 2) for val in df['product_length_cm'].dropna()))
# product_height_cm = sorted(set(round(val, 2) for val in df['product_height_cm'].dropna()))
# product_width_cm = sorted(set(round(val, 2) for val in df['product_width_cm'].dropna()))



# 'total_order_value', 'order_item_id', 'purchase_date', 'purchase_time',

product_category_name = list(df['product_category_name'].unique())

# 'purchase_date', 'purchase_time', order_item_id

# the purchase date shall be written manually by the user

purchase_time = list(df['purchase_time'].unique())
order_item_id = list(df['order_item_id'].unique())



seller_state = list(df['seller_state'].unique())
customer_state = list(df['customer_state'].unique())
seller_city = list(df['seller_city'].unique())
customer_city = list(df['customer_city'].unique())


import ipywidgets as widgets
from IPython.display import display, clear_output,HTML
from datetime import date

# Create input fields

product_length_cm = widgets.FloatText(value=28.0, description="product_length_cm:" ,
    style={'description_width': 'initial'})


product_height_cm = widgets.FloatText(value=9.0, description="product_height_cm:" ,
    style={'description_width': 'initial'})


product_width_cm = widgets.FloatText(value=14.0, description="product_width_cm:" ,
    style={'description_width': 'initial'})

# 1
product_category_name = widgets.Dropdown(value=product_category_name[0], options=product_category_name ,description="product_category_name:", layout=widgets.Layout(width='400px'),
    style={'description_width': 'initial'})
 
# 3
seller_state = widgets.Dropdown(value=seller_state[0], options=seller_state ,description="seller_state:"
                               , layout=widgets.Layout(width='400px'),
    style={'description_width': 'initial'})


# 'total_order_value', 'order_item_id', 'purchase_date', 'purchase_time',



# Time picker substitute (use Text for simplicity if TimePicker isn't available)
purchase_time = widgets.Text(
    description='Purchase Time (HH:MM):',
    placeholder='e.g., 14:30',
    style={'description_width': 'initial'}
)



order_item_id = widgets.Dropdown(value=order_item_id[0], options=order_item_id ,description="Number of items:"
                                 , layout=widgets.Layout(width='400px'),
    style={'description_width': 'initial'})


customer_state = widgets.Dropdown(value=customer_state[0], options=customer_state ,description="customer_state:"
                                 , layout=widgets.Layout(width='400px'),
    style={'description_width': 'initial'})

seller_city = widgets.Dropdown(value=seller_city[0], options=seller_city ,description="seller_city:" , layout=widgets.Layout(width='400px'),
    style={'description_width': 'initial'})

customer_city = widgets.Dropdown(value=customer_city[0], options=customer_city ,description="customer_city:" , layout=widgets.Layout(width='400px'),
    style={'description_width': 'initial'})


# 4
freight_value = widgets.FloatText(value=13.29, description="freight_value:" ,
    style={'description_width': 'initial'})

# 'price', 'product_volume_cm3' -> backend 😊


price = widgets.FloatText(value=58.9, description="price:" ,
    style={'description_width': 'initial'})

total_order_value = widgets.FloatText(value=100, description="total_order_value:" ,
    style={'description_width': 'initial'})


product_weight_g = widgets.FloatText(value=650.0, description="product_weight_g:" ,
    style={'description_width': 'initial'})

purchase_date = widgets.DatePicker(
    description='Purchase Date:',
    disabled=False,
    value=date.today(),
    style={'description_width': 'initial'}
)


# 'total_order_value', 'order_item_id', 'purchase_date', 'purchase_time',

# Display widgets
display( 
    product_category_name,
    
    product_weight_g, product_length_cm,product_height_cm, product_width_cm ,
    
    seller_state, customer_state, seller_city, customer_city, 
    
    freight_value, price,
    
    order_item_id, purchase_date, purchase_time
    

)

# Output widget for displaying updates
output = widgets.Output()
display(output)

data = {}


# Function to save values
def save_changes(b):
    global data 
    
    data = {
    "customer_state": customer_state.value,
    "product_weight_g": product_weight_g.value,
    "shipping_duration_days": 0,
    "price": price.value,
    "product_volume_cm3": product_length_cm.value * product_height_cm.value * product_width_cm.value,
    "seller_state": seller_state.value,
    "customer_city": customer_city.value,
    "product_height_cm": product_height_cm.value,
    "seller_city": seller_city.value,
    "product_length_cm": product_length_cm.value,
    "product_category_name": product_category_name.value,
    "product_width_cm": product_width_cm.value,
    "order_item_id": order_item_id.value,
    "freight_value": freight_value.value,
    "purchase_date": purchase_date.value,
    "purchase_time": purchase_time.value,
    "total_order_value": (float(price.value) * int(order_item_id.value)) + (float(freight_value.value) * int(order_item_id.value))
    }
    
    


    
    
# Save button
save_button = widgets.Button(description="Save", button_style='success')
save_button.on_click(save_changes)
display(save_button)

Dropdown(description='product_category_name:', layout=Layout(width='400px'), options=('cool_stuff', 'pet_shop'…

FloatText(value=650.0, description='product_weight_g:', style=DescriptionStyle(description_width='initial'))

FloatText(value=28.0, description='product_length_cm:', style=DescriptionStyle(description_width='initial'))

FloatText(value=9.0, description='product_height_cm:', style=DescriptionStyle(description_width='initial'))

FloatText(value=14.0, description='product_width_cm:', style=DescriptionStyle(description_width='initial'))

...

Dropdown(description='Number of items:', layout=Layout(width='400px'), options=(1, 2, 3, 4, 5, 6, 7, 8, 9, 10,…

DatePicker(value=datetime.date(2025, 4, 10), description='Purchase Date:', style=DescriptionStyle(description_…

Text(value='', description='Purchase Time (HH:MM):', placeholder='e.g., 14:30', style=DescriptionStyle(descrip…

Output()

Button(button_style='success', description='Save', style=ButtonStyle())

In [11]:
reload()

In [12]:
df.head()

,price,product_weight_g,total_order_value,product_length_cm,product_height_cm,product_width_cm,order_item_id,product_category_name,seller_state,customer_state,seller_city,customer_city,freight_value,purchase_date,purchase_time,shipping_duration_days,product_volume_cm3
0,58.90,650.0,72.19,28.0,9.0,14.0,1,cool_stuff,SP,RJ,volta redonda,campos dos goytacazes,13.29,2017-09-13,8,7,163800.0
1,239.90,30000.0,259.83,50.0,30.0,40.0,1,pet_shop,SP,SP,sao paulo,santa fe do sul,19.93,2017-04-26,10,16,45000000.0
2,199.00,3050.0,216.87,33.0,13.0,33.0,1,furniture_decor,MG,MG,borda da mata,para de minas,17.87,2018-01-14,14,7,1308450.0
3,12.99,200.0,25.78,16.0,10.0,15.0,1,perfumery,SP,SP,franca,atibaia,12.79,2018-08-08,10,6,32000.0
4,199.90,3750.0,218.04,35.0,40.0,30.0,1,garden_tools,PR,SP,loanda,varzea paulista,18.14,2017-02-04,13,25,5250000.0


In [159]:
df

,customer_state,product_weight_g,shipping_duration_days,price,product_volume_cm3,seller_state,customer_city,product_height_cm,seller_city,product_length_cm,product_category_name,product_width_cm,order_item_id,freight_value,purchase_date,purchase_time,total_order_value
0,RJ,650.0,0,58.9,3528.0,SP,campos dos goytacazes,9.0,volta redonda,28.0,cool_stuff,14.0,1,13.29,2025-04-10,9,72.19


In [24]:
data

{'customer_state': 'RJ',
 'product_weight_g': 650.0,
 'shipping_duration_days': 0,
 'price': 58.9,
 'product_volume_cm3': 3528.0,
 'seller_state': 'SP',
 'customer_city': 'campos dos goytacazes',
 'product_height_cm': 9.0,
 'seller_city': 'volta redonda',
 'product_length_cm': 28.0,
 'product_category_name': 'cool_stuff',
 'product_width_cm': 14.0,
 'order_item_id': 1,
 'freight_value': 13.29,
 'purchase_date': datetime.date(2017, 9, 13),
 'purchase_time': '8',
 'total_order_value': 72.19}

# click SAVE BEFORE RUNNING 📁

In [25]:
df = pd.DataFrame([data])

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

pred = final_one_prediction(df,rf,xgbR,nn)

# Create HTML with styling for a card
html_content = f"""
<div style="border: 2px solid #4CAF50; border-radius: 10px; padding: 20px; width: 300px; 
            background-color: #f0f8ff; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.1);">
    <h2 style="color: green; text-align: center;">Predicted Days: <strong>{pred[0]:.2f} Days</strong></h2>
</div>
"""

# Display the card
display(HTML(html_content))

# Error Interpretation

In [26]:
log_rmse = 0.42
error_factor = np.exp(log_rmse)
percent_error = (error_factor - 1) * 100

error_factor, percent_error

(1.5219615556186337, 52.196155561863364)

In [27]:
prediction = 13
percent_error = 52.20 / 100

lower_bound = prediction * (1 - percent_error)
upper_bound = prediction * (1 + percent_error)

lower_bound, upper_bound

(6.2139999999999995, 19.786)

In [28]:
13 - 7

6

In [29]:
# WHICH ESTIMATE IS BETTER ???


df = load_df_and_clean()

df['shipping_duration_days'] = 0
df.isna().sum()

ans = final_one_prediction(df,rf,xgbR, nn)

my_estimation = pd.Series(ans)

my_estimation

df = load_df_and_clean()

df = df.reset_index().drop(columns=['index'])

df['my_estimation'] = my_estimation

olist_estimate = pd.read_csv('estimated_series.csv')

df['olist_estimate'] = olist_estimate


import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Assuming df is your DataFrame
y_true = df['shipping_duration_days']
my_pred = df['my_estimation']
olist_pred = df['olist_estimate']

# Calculate MAE
mae_my = mean_absolute_error(y_true, my_pred)
mae_olist = mean_absolute_error(y_true, olist_pred)

# Calculate RMSE
rmse_my = mean_squared_error(y_true, my_pred, squared=False)
rmse_olist = mean_squared_error(y_true, olist_pred, squared=False)

# Print results
print(f"📦 My Estimation - MAE: {mae_my:.3f}, RMSE: {rmse_my:.3f}")
print(f"🏷️ Olist Estimate - MAE: {mae_olist:.3f}, RMSE: {rmse_olist:.3f}")

📦 My Estimation - MAE: 6.204, RMSE: 8.426
🏷️ Olist Estimate - MAE: 12.913, RMSE: 15.304


In [ ]:
df[[ 'shipping_duration_days', 'my_estimation', 'olist_estimate' ]].to_csv('estimate_olist_vs_our_estimate.csv',index=False)


# `Recommendation 1:`

## FROM THE Freight Cost 30.8 % WE CAN UNDERSTAND


##  Negotiate Better Shipping Rates
 - Work with shipping companies to get better rates, especially for large volumes. (such as DHL)
 - This helps reduce overall shipping costs and speed up delivery.
 
##  Get help from locals
 - search for local people to deliver products in the same city, instead of large shipping companies
 

# 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 🥣 

# `Recommendation 2:`
# Targeted Distribution Based on Location (customer_state - 20.9%, customer_city - 7.6%, seller_state & seller_city - 10.5%)

## Insight: The customer’s and seller’s geographic locations `affects` the `delivery time.` 🏎️🕐
# Recommendations:
### -  Group sellers geographically and route orders from nearest sellers to the customer.
### - Encourage sellers to relocate or store inventory in busy city areas with high order volume.
### - Use location-based order predictions to pre-position inventory. (Machine Learning Project)